# 12 — Final multi-criteria Earth ranking

This notebook combines validated process-specific Earth candidate tables into one transparent priority ranking. It is not a supervised classifier. It preserves each process score, applies explicit priorities, penalizes missing evidence, and reports sensitivity to alternative priorities. Unsupported or inconclusive processes are ignored rather than assigned fabricated scores.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = ROOT / 'final_multi_criteria_ranking'; OUT.mkdir(exist_ok=True)
# Change these only when the process has passed its Mars evidence gate.
PROCESS_PRIORITIES = {
    'fine_particulate_dust_transport': 1.0,
    'erosion_sediment_transport': 0.0,
    'volcanic_basaltic_terrain': 0.0,
    'ice_permafrost_dynamics': 0.0,
    'water_driven_alteration': 0.0,
    'impact_crater_terrain': 0.0,
}
files = list(ROOT.rglob('*earth_candidates.parquet'))
# Notebook 09 is the verified dust workflow and uses this explicit filename.
dust_refined = list(ROOT.rglob('refined_earth_matches.parquet'))
files.extend(dust_refined)
if not files:
    raise FileNotFoundError('No process Earth candidate tables found. Run a validated process matcher first.')
frames=[]
for path in files:
    process = path.name.replace('_earth_candidates.parquet','')
    if path.name == 'refined_earth_matches.parquet':
        process = 'fine_particulate_dust_transport'
    if PROCESS_PRIORITIES.get(process, 0.0) <= 0: continue
    frame=pd.read_parquet(path).copy()
    if not {'latitude','longitude','analog_score'}.issubset(frame.columns): continue
    frame['process_id']=process; frame['process_score']=pd.to_numeric(frame['analog_score'],errors='coerce').clip(0,1)
    frames.append(frame[['latitude','longitude','process_id','process_score']])
if not frames:
    raise ValueError('No enabled process has a valid Earth candidate table. Enable only supported processes.')
long=pd.concat(frames,ignore_index=True)
pivot=long.pivot_table(index=['latitude','longitude'],columns='process_id',values='process_score',aggfunc='max')
weights=pd.Series({k:v for k,v in PROCESS_PRIORITIES.items() if v>0},dtype=float); weights/=weights.sum()
used=[p for p in weights.index if p in pivot.columns]
if not used: raise ValueError('Enabled priorities do not match available process tables')
coverage=pivot[used].notna().dot(weights[used])
score=pivot[used].fillna(0).dot(weights[used])
ranked=pivot.reset_index(); ranked['priority_score']=score.to_numpy(); ranked['evidence_coverage']=coverage.to_numpy(); ranked['missing_evidence_penalty']=1-ranked.evidence_coverage; ranked['rank']=ranked.priority_score.rank(method='first',ascending=False).astype(int); ranked=ranked.sort_values('rank')
ranked.to_parquet(OUT/'final_ranked_earth_candidates.parquet',index=False); ranked.to_csv(OUT/'final_ranked_earth_candidates.csv',index=False)
scenarios={'equal':{p:1 for p in used},'dust_heavy':{p:(3 if 'dust' in p else 1) for p in used}}
sens=[]
for name,w in scenarios.items():
    w=pd.Series(w,dtype=float); w/=w.sum(); s=pivot[used].fillna(0).dot(w[used])
    sens.append(pd.DataFrame({'latitude':pivot.index.get_level_values(0),'longitude':pivot.index.get_level_values(1),'scenario':name,'score':s.to_numpy(),'rank':s.rank(method='first',ascending=False).astype(int)}))
pd.concat(sens,ignore_index=True).to_csv(OUT/'priority_sensitivity.csv',index=False)
summary={'enabled_processes':used,'weights':weights.to_dict(),'candidate_count':int(len(ranked)),'top_20':ranked.head(20).to_dict('records'),'interpretation':'Overall priority ranking of Earth regions across enabled validated process scores; not proof of identical planetary environments.'}
(OUT/'final_ranking_summary.json').write_text(json.dumps(summary,indent=2,default=float),encoding='utf-8')
display(ranked.head(20)); print(json.dumps(summary,indent=2,default=float))
